# DarkTech Remote Worker (Colab A100)

Runs the audio-generation worker on the Colab GPU and exposes it via Cloudflare Tunnel
with a stable public HTTPS URL.

Your local Gradio UI (running in VSCode/your machine) hits this URL whenever it needs
to render a stem. Mastering, orchestration with DeepSeek, and the UI itself stay local.

**Setup before running**: Runtime > Change runtime type > GPU > A100 (Colab Pro).

In [ ]:
import os
import secrets

# Generate or paste an API key. The same key must go in your local .env as
# DARKTECH_REMOTE_API_KEY. Default: a fresh random key per session.
os.environ['DARKTECH_WORKER_API_KEY'] = secrets.token_urlsafe(24)
print('API key for this session:')
print('   ', os.environ['DARKTECH_WORKER_API_KEY'])
print()
print('Copy that into your local .env as DARKTECH_REMOTE_API_KEY')

os.environ.setdefault('HF_TOKEN', '')  # optional, raises HF download rate limits

In [ ]:
# Install the package and the worker-only dependencies.
REPO_URL = 'https://github.com/whorning/gorit-lab-darktech-generator.git'
!git clone -q $REPO_URL /content/gorit_lab_darktech_generator || (cd /content/gorit_lab_darktech_generator && git pull -q)
%cd /content/gorit_lab_darktech_generator
!pip install -q -e '.[ml]' fastapi 'uvicorn[standard]' 2>&1 | tail -5

# Mount Drive so model weights persist across Colab sessions.
from darktech_generator.colab.bootstrap import bootstrap
import json
print(json.dumps(bootstrap(), indent=2))

# Install cloudflared if not present.
import shutil, subprocess
if not shutil.which('cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /tmp/cf.deb
    !sudo dpkg -i /tmp/cf.deb 2>&1 | tail -2

In [ ]:
# Launch the worker + a Cloudflare quick tunnel. The tunnel URL is printed below.
# Paste it in your local .env as DARKTECH_REMOTE_URL=https://...
#
# This cell blocks until you stop it (or until Colab disconnects). Leave it running
# while you use the local UI.

import subprocess, threading, time, re, sys
from pathlib import Path

# 1) Start the worker in the background.
worker_log = Path('/tmp/worker.log')
worker = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn',
     'darktech_generator.generation.remote_worker:build_app',
     '--factory', '--host', '0.0.0.0', '--port', '8000'],
    stdout=worker_log.open('w'), stderr=subprocess.STDOUT,
)
time.sleep(3)
print('Worker PID', worker.pid, '(logs at /tmp/worker.log)')

# 2) Start a Cloudflare quick tunnel and capture the URL it prints.
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

public_url = None
url_re = re.compile(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com')
for line in iter(tunnel.stdout.readline, ''):
    print(line, end='')
    m = url_re.search(line)
    if m and public_url is None:
        public_url = m.group(0)
        print('\n=====================================================')
        print(' Cloudflare Tunnel ready')
        print(' Set on your local machine:')
        print(f'   DARKTECH_REMOTE_URL={public_url}')
        print('=====================================================\n')